# Baseline unary IND discovery on Sakila: De Marchi's algorithm

In [1]:
import sqlite3  
import urllib.request  
import pandas as pd  

In [2]:
url = "https://sq.io/testdata/sakila.db"  
urllib.request.urlretrieve(url, "sakila.db")  
print("Database Downloaded Successfully !!")  

Database Downloaded Successfully !!


In [3]:
#open database file and display all tables
con = sqlite3.connect("sakila.db")  

tables = [r[0] for r in con.execute(  
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")]  
print(len(tables), "tables:") 
print(tables)  

16 tables:
['actor', 'country', 'city', 'address', 'language', 'category', 'customer', 'film', 'film_actor', 'film_category', 'film_text', 'inventory', 'staff', 'store', 'payment', 'rental']


In [7]:
import sqlite3  
con = sqlite3.connect("sakila.db")  
print(con.execute("SELECT COUNT(*) FROM film_text").fetchone()[0]) # number of rows in film_text

0


In [8]:
pd.read_sql("SELECT * FROM actor LIMIT 10", con)  

,actor_id,first_name,last_name,last_update
0,1,PENELOPE,GUINESS,2020-02-15 06:59:28
1,2,NICK,WAHLBERG,2020-02-15 06:59:28
2,3,ED,CHASE,2020-02-15 06:59:28
3,4,JENNIFER,DAVIS,2020-02-15 06:59:28
4,5,JOHNNY,LOLLOBRIGIDA,2020-02-15 06:59:28
5,6,BETTE,NICHOLSON,2020-02-15 06:59:28
6,7,GRACE,MOSTEL,2020-02-15 06:59:28
7,8,MATTHEW,JOHANSSON,2020-02-15 06:59:28
8,9,JOE,SWANK,2020-02-15 06:59:28
9,10,CHRISTIAN,GABLE,2020-02-15 06:59:28


## Step 1: every column becomes a set of distinct values

In [9]:
def norm(v):  
    if isinstance(v, float) and v.is_integer():  
        v = int(v)  
    return str(v)  

cols = {}  
for t in tables:  
    df = pd.read_sql(f'SELECT * FROM "{t}"', con) 
    for c in df.columns:  
        vals = set(map(norm, df[c].dropna()))  # distinct values as text; dropna() ignores empty cells (NULL)
        if vals:  # only keep columns that have at least one value
            cols[(t, c)] = vals  

print(len(cols), "Columns Loaded Successfully") 

83 Columns Loaded Successfully


## Step 2: De Marchi's algorithm
1. Build an **inverted index**: for every value, which columns contain it?
2. For every column A, intersect the column lists of all of A's values. The columns that are left contain
   every value of A, so A ⊆ B.

In [11]:
def de_marchi(cols):  
    index = {} 
    for col in cols: 
        for v in cols[col]:  
            if v not in index: 
                index[v] = set() 
            index[v].add(col) 

    found = set()  # the INDs we find will be collected here
    for a in cols: 
        refs = set(cols)  
        for v in cols[a]: 
            refs = refs & index[v]  
        refs.discard(a) 
        for b in refs:  
            if a[0] != b[0]: 
                found.add((a, b))  
    return found  # give back all INDs

found = de_marchi(cols)  
print(len(found), "INDs Found") 

596 INDs Found


## Step 3: Ground truth = the foreign keys declared in the database

In [12]:
def primary_key(t):  
    return [r[1] for r in con.execute(f'PRAGMA table_info("{t}")') if r[5] >= 1]  

gt = set()  # the declared foreign keys will be collected here
for t in tables: 
    for r in con.execute(f'PRAGMA foreign_key_list("{t}")'):  
        ref_table, from_col, to_col = r[2], r[3], r[4] 
        if to_col is None:   # sometimes only the primary key is meant, without naming it
            pk = primary_key(ref_table)  # then look up the primary key of the referenced table
            to_col = pk[0] if pk else None  
        if to_col and ref_table != t:  # skip foreign keys that point to the same table
            gt.add(((t, from_col), (ref_table, to_col)))  
            
print(len(gt), "declared FKs")  
sorted(gt) 

22 declared FKs


[(('address', 'city_id'), ('city', 'city_id')),
 (('city', 'country_id'), ('country', 'country_id')),
 (('customer', 'address_id'), ('address', 'address_id')),
 (('customer', 'store_id'), ('store', 'store_id')),
 (('film', 'language_id'), ('language', 'language_id')),
 (('film', 'original_language_id'), ('language', 'language_id')),
 (('film_actor', 'actor_id'), ('actor', 'actor_id')),
 (('film_actor', 'film_id'), ('film', 'film_id')),
 (('film_category', 'category_id'), ('category', 'category_id')),
 (('film_category', 'film_id'), ('film', 'film_id')),
 (('inventory', 'film_id'), ('film', 'film_id')),
 (('inventory', 'store_id'), ('store', 'store_id')),
 (('payment', 'customer_id'), ('customer', 'customer_id')),
 (('payment', 'rental_id'), ('rental', 'rental_id')),
 (('payment', 'staff_id'), ('staff', 'staff_id')),
 (('rental', 'customer_id'), ('customer', 'customer_id')),
 (('rental', 'inventory_id'), ('inventory', 'inventory_id')),
 (('rental', 'staff_id'), ('staff', 'staff_id')),
 

## Step 4: Evaluation

In [14]:
tp = found & gt  # INDs that we found AND that are declared foreign keys
fp = found - gt  # INDs that we found but that are NOT declared foreign keys
fn = gt - found  # declared foreign keys that we did NOT find

tn = 0  # counter for true negatives: pairs we did not report and that are not real foreign keys
for a in cols:  
    for b in cols:  
        if a[0] != b[0]:  
            if (a, b) not in found and (a, b) not in gt:  # not reported AND not a real foreign key?
                tn = tn + 1  # then it is a true negative

precision = len(tp) / len(found)  # of all INDs we found, how many are real foreign keys?
recall = len(tp) / len(gt)  # of all real foreign keys, how many did we find?
f1 = 2 * precision * recall / (precision + recall)  

print("Declared FKs         :", len(gt))  
print("Caught by De Marchi  :", len(tp), f"(recall = {recall:.2f})")  
print("Missed FKs           :", sorted(fn)) 
print("Extra INDs (not FK)  :", len(fp)) 
print(f"Precision = {precision:.3f}   Recall = {recall:.3f}   F1 = {f1:.3f}")  

#confusion matrix
matrix = pd.DataFrame( 
    [["TP = " + str(len(tp)), "FP = " + str(len(fp))],  
     ["FN = " + str(len(fn)), "TN = " + str(tn)]], 
    index=["Found as IND", "Not found"], 
    columns=["Declared foreign key", "Not a declared foreign key"])  
print("\nConfusion matrix:") 
display(matrix) 

Declared FKs         : 22
Caught by De Marchi  : 21 (recall = 0.95)
Missed FKs           : [(('film', 'original_language_id'), ('language', 'language_id'))]
Extra INDs (not FK)  : 575
Precision = 0.035   Recall = 0.955   F1 = 0.068

Confusion matrix:


,Declared foreign key,Not a declared foreign key
Found as IND,TP = 21,FP = 575
Not found,FN = 1,TN = 5712


In [15]:
all_pairs = 0  
tn = 0  
for a in cols:  
    for b in cols: 
        if a[0] != b[0]:  
            all_pairs = all_pairs + 1  #
            if (a, b) not in found and (a, b) not in gt: 
                tn = tn + 1  
print("All candidate pairs :", all_pairs) 
print("True negatives      :", tn)  
print("Accuracy            :", round((len(tp) + tn) / all_pairs, 3))  # (TP + TN) / all pairs

All candidate pairs : 6308
True negatives      : 5712
Accuracy            : 0.909


In [16]:
missing = [c for pair in gt for c in pair if c not in cols]  # foreign key columns that were not kept
print("Foreign key columns without any values (all NULL):", sorted(set(missing)))  # why a foreign key can be missed

Foreign key columns without any values (all NULL): [('film', 'original_language_id')]
